<a href="https://colab.research.google.com/github/LsYoli/fisica_computacional/blob/main/Taller_7/Taller7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Preparación: bibliotecas de Python y colores del curso
import numpy as np
import matplotlib.pyplot as plt

AZUL, DORADO, ROJO, VERDE, GRIS = "#162A54", "#C9972C", "#B02A20", "#2E7D32", "#7F7F7F"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# Comprobación del compilador de Fortran
!gfortran --version | head -n 1

GNU Fortran (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## 5. Taller 7: energía disipada y derivadas de un oscilador amortiguado, y el valor de $\pi$

### A. Enunciado y requisitos

Un sensor registra la posición de un oscilador amortiguado:

$$x(t) = e^{-\gamma t}\cos(\omega t),$$

donde $x$ es la posición [m] (amplitud inicial 1 m), $t$ el tiempo [s], $\gamma = 0.5\ \text{s}^{-1}$ el coeficiente de amortiguamiento, que fija la rapidez con que decae la amplitud, y $\omega = 2\pi\ \text{rad/s}$ la frecuencia angular de oscilación (periodo de 1 s). Esta posición es solución de la ecuación del oscilador amortiguado

$$\ddot{x} + 2\gamma\,\dot{x} + \omega_0^2\,x = 0, \qquad \omega_0^2 = \omega^2 + \gamma^2,$$

donde $2\gamma\dot{x}$ es la fuerza de fricción por unidad de masa, proporcional a la velocidad, y $\omega_0$ es la frecuencia natural del oscilador sin amortiguamiento. La velocidad y la aceleración exactas son

$$v(t) = -e^{-\gamma t}\left[\gamma\cos(\omega t) + \omega\sin(\omega t)\right], \qquad a(t) = e^{-\gamma t}\left[(\gamma^2 - \omega^2)\cos(\omega t) + 2\gamma\omega\sin(\omega t)\right].$$

**Energía disipada.** La energía mecánica por unidad de masa [J/kg] es

$$E(t) = \underbrace{\tfrac{1}{2}v^2}_{\text{cinética}} + \underbrace{\tfrac{1}{2}\omega_0^2 x^2}_{\text{potencial elástica}} .$$

Derivando y usando la ecuación del oscilador se obtiene $\frac{dE}{dt} = -2\gamma v^2$: la fricción extrae energía a un ritmo $2\gamma v^2$. Integrando entre $0$ y $T$,

$$E_{\text{dis}} = \int_0^T 2\gamma\,v(t)^2\,dt = E(0) - E(T),$$

donde $E_{\text{dis}}$ es la energía disipada por la fricción en el intervalo $[0, T]$. El lado derecho se calcula directamente con $x$ y $v$ en los dos extremos; el lado izquierdo requiere una integral numérica. Que coincidan es una prueba de la conservación de la energía.

**El taller tiene tres partes:**

| Parte | Lenguaje | Qué se hace |
|---|---|---|
| B | Fortran | Módulo de integración de Romberg; energía disipada en $T = 3$ s |
| C | Python | Velocidad y aceleración a partir de las posiciones, con varias fórmulas, comparadas con las exactas |
| D | Fortran | El mismo módulo de Romberg aplicado a $\pi = \int_0^1 \frac{4}{1+x^2}\,dx$, comparado con Monte Carlo (Taller 5) |

**Requisitos generales**

1. Todo el trabajo se guarda en una carpeta llamada `taller_oscilador_romberg`, con los nombres de archivo que se indican en cada parte.
2. Se usa doble precisión: `real(dp)` en Fortran y `float` (64 bits) en Python.
3. La tabla de Romberg se almacena en un arreglo `allocatable` de dos dimensiones con índices desde 0.
4. Las derivadas se calculan sobre el arreglo completo con rebanadas, sin ciclos `for` sobre los puntos.
5. Se entrega el pseudocódigo marco de la Parte B con sus tres espacios completos.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**No está permitido** usar `np.gradient`, `np.trapz`, `np.trapezoid`, `scipy.integrate` ni ninguna función de biblioteca que integre o derive. Las fórmulas se programan a partir de las secciones 3 y 4. Cada estudiante sustentará su programa.

</div>

**Pseudocódigo marco de la integración de Romberg** (se entrega con los espacios completos)

```
ENTRADAS: a, b, tol, kmax
h      ← b − a          (Δx: un solo subintervalo en el nivel 0)
S(0,0) ← (h/2)·[ f(a) + f(b) ]
n_eval ← 2
PARA k = 1, 2, ..., kmax:
    h ← h/2
    suma ← Σ f(a + (2i − 1)·h),  i = 1, ..., 2^(k−1)
    n_eval ← n_eval + 2^(k−1)
    S(k,0) ← S(k−1,0)/2 + h·suma
    PARA j = 1, ..., k:
        S(k,j) ← ( 4^j · S(k,j−1) − S(k−1,j−1) ) / ( 4^j − 1 )
    FIN PARA
    SI |S(k,k) − S(k−1,k−1)| < tol: TERMINAR con k_final = k
FIN PARA
SALIDAS: tabla S, k_final, n_eval
```

### B. Parte Fortran: energía disipada con integración de Romberg

**Actividades**

1. Complete el pseudocódigo marco. Los tres espacios se deducen de la sección 3.4 y de su nota sobre el refinamiento del trapecio.
2. Escriba el módulo `modulo_romberg.f90` con la integración de Romberg y los integrandos, y el programa `programa_energia.f90` que lo usa. La organización en procedimientos es decisión suya; debe respetar las reglas de la Semana 6 (`intent` en todos los argumentos, procedimientos en el módulo). El mismo módulo se usará sin cambios en la Parte D.
3. Calcule $E_{\text{dis}}$ en $[0, 3]$ s con tolerancia $10^{-6}$ y el criterio de parada de la diagonal, contando las evaluaciones del integrando.
4. Calcule $E(0) - E(T)$ con las fórmulas de $x$ y $v$, y compárelo con $E_{\text{dis}}$.
5. Escriba la tabla en `tabla_energia.dat` y el resultado en `resultado_energia.dat`. Compile con `-fcheck=bounds`.

**Qué preparar**

| Archivo | Contenido |
|---|---|
| `modulo_romberg.f90` | Integración de Romberg con conteo de evaluaciones; integrandos $2\gamma v^2$ (Parte B) y $4/(1+x^2)$ (Parte D) |
| `programa_energia.f90` | Usa el módulo; escribe los dos archivos `.dat` de esta parte |
| `tabla_energia.dat` | Una fila por nivel $k = 0, \dots, k_{\text{final}}$, con $S_{k,0}, \dots, S_{k,k}$ |
| `resultado_energia.dat` | Una línea: $k_{\text{final}}$, número de evaluaciones, $E_{\text{dis}}$ (Romberg), $E(0) - E(T)$, diferencia absoluta |

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** Pasar una función como argumento de otra requiere bloques `interface`, que se ven en el Capítulo 4. Por ahora, el módulo puede contener los dos integrandos y una función que elija entre ellos según un argumento entero; así la integración de Romberg se escribe una sola vez y sirve para las Partes B y D.

</div>

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** Cuente las evaluaciones del integrando dentro del procedimiento que lo llama, no a mano. La tabla hasta el nivel $k$ debe usar exactamente $2^k + 1$ evaluaciones; si su programa reporta más, está evaluando la función dos veces en algunos puntos: revise el espacio 1 del pseudocódigo.

</div>

In [5]:
%%writefile modulo_romberg.f90
! ============================================================
! Módulo:    modulo_romberg
! Propósito: integración de Romberg con conteo de evaluaciones
!            e integrandos del Taller 7.
! ============================================================
module modulo_romberg
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  real(dp), parameter :: pi = 4*atan(1.0_dp)
  real(dp), parameter :: gamma = 0.5_dp          ! amortiguamiento [1/s]
  real(dp), parameter :: omega = 2*pi            ! frecuencia angular [rad/s]
  real(dp), parameter :: omega0_2 = omega**2 + gamma**2
  integer, parameter :: ID_ENERGIA = 1           ! integrando 2*gamma*v(t)^2
  integer, parameter :: ID_PI = 2                ! integrando 4/(1+x^2)
contains

  ! ----------------------------------------------------------
  ! posicion: x(t) = exp(-gamma t) cos(omega t)
  ! ----------------------------------------------------------
  real(dp) function posicion(t)
    real(dp), intent(in) :: t
    posicion = exp(-gamma*t)*cos(omega*t)
  end function posicion

  ! ----------------------------------------------------------
  ! velocidad: v(t) exacta del oscilador amortiguado
  ! ----------------------------------------------------------
  real(dp) function velocidad(t)
    real(dp), intent(in) :: t
    velocidad = -exp(-gamma*t)*(gamma*cos(omega*t) + omega*sin(omega*t))
  end function velocidad

  ! ----------------------------------------------------------
  ! energia: E(t) = v^2/2 + omega0^2 x^2/2   [J/kg]
  ! ----------------------------------------------------------
  real(dp) function energia(t)
    real(dp), intent(in) :: t
    energia = 0.5_dp*velocidad(t)**2 + 0.5_dp*omega0_2*posicion(t)**2
  end function energia

  ! ----------------------------------------------------------
  ! integrando: elige el integrando según id
  ! Entradas: id (ID_ENERGIA o ID_PI), x
  ! ----------------------------------------------------------
  real(dp) function integrando(id, x)
    integer, intent(in)  :: id
    real(dp), intent(in) :: x
    select case (id)
    case (ID_ENERGIA)
      integrando = 2*gamma*velocidad(x)**2
    case (ID_PI)
      integrando = 4.0_dp/(1.0_dp + x*x)
    case default
      error stop 'integrando: id desconocido'
    end select
  end function integrando

  ! ----------------------------------------------------------
  ! romberg: integración de Romberg de f(id, x) en [a, b]
  ! Entradas: id, a, b, tol, kmax
  ! Salidas:  S(0:kmax,0:kmax) tabla (solo el triángulo 0..k_final),
  !           k_final nivel alcanzado, n_eval evaluaciones de f
  ! ----------------------------------------------------------
  subroutine romberg(id, a, b, tol, kmax, S, k_final, n_eval)
    integer, intent(in)  :: id, kmax
    real(dp), intent(in) :: a, b, tol
    real(dp), allocatable, intent(out) :: S(:,:)
    integer, intent(out) :: k_final, n_eval
    real(dp) :: h, suma
    integer :: k, j, i

    allocate(S(0:kmax, 0:kmax))
    S = 0.0_dp
    h = b - a
    S(0,0) = 0.5_dp*h*(integrando(id, a) + integrando(id, b))
    n_eval = 2
    k_final = 0
    do k = 1, kmax
       h = h/2
       ! puntos nuevos del nivel k: a + (2i-1)h, i = 1..2^(k-1)
       suma = 0.0_dp
       do i = 1, 2**(k-1)
          suma = suma + integrando(id, a + (2*i - 1)*h)
       end do
       n_eval = n_eval + 2**(k-1)
       S(k,0) = 0.5_dp*S(k-1,0) + h*suma
       do j = 1, k
          S(k,j) = (4.0_dp**j*S(k,j-1) - S(k-1,j-1))/(4.0_dp**j - 1.0_dp)
       end do
       k_final = k
       if (abs(S(k,k) - S(k-1,k-1)) < tol) exit
    end do
  end subroutine romberg

end module modulo_romberg

Writing modulo_romberg.f90


In [6]:
%%writefile programa_energia.f90
! ============================================================
! Programa:  programa_energia
! Propósito: energía disipada por un oscilador amortiguado en
!            [0, 3] s con Romberg, y comparación con E(0) - E(T).
! Salida:    tabla_energia.dat, resultado_energia.dat
! ============================================================
program programa_energia
  use modulo_romberg, only: dp, ID_ENERGIA, romberg, energia
  implicit none
  real(dp), parameter :: T = 3.0_dp, tol = 1.0e-6_dp
  integer, parameter :: kmax = 20
  real(dp), allocatable :: S(:,:)
  real(dp) :: e_dis, e_ref
  integer :: k, k_final, n_eval, u

  call romberg(ID_ENERGIA, 0.0_dp, T, tol, kmax, S, k_final, n_eval)
  e_dis = S(k_final, k_final)
  e_ref = energia(0.0_dp) - energia(T)

  open(newunit=u, file='tabla_energia.dat', status='replace')
  do k = 0, k_final
     write(u, '(*(es24.15e3,1x))') S(k, 0:k)
  end do
  close(u)

  open(newunit=u, file='resultado_energia.dat', status='replace')
  write(u, '(i4,1x,i6,3(1x,es24.15e3))') k_final, n_eval, e_dis, e_ref, abs(e_dis - e_ref)
  close(u)

  print '(a,i3,a,i5)', 'k_final = ', k_final, '   evaluaciones = ', n_eval
  print '(a,f16.10)', 'E_dis (Romberg) = ', e_dis
  print '(a,f16.10)', 'E(0) - E(T)     = ', e_ref
  print '(a,es10.2)', '|diferencia|    = ', abs(e_dis - e_ref)
end program programa_energia

Writing programa_energia.f90


In [7]:
# Compila y ejecuta la Parte B (ejecute esta celda cuando sus archivos estén listos)
!gfortran -fcheck=bounds -o programa_energia modulo_romberg.f90 programa_energia.f90 && ./programa_energia

k_final =   8   evaluaciones =   257
E_dis (Romberg) =    18.9940046974
E(0) - E(T)     =    18.9940046969
|diferencia|    =   4.53E-10


### C. Parte Python: velocidad y aceleración a partir de las posiciones

**Actividades**

1. Escriba `derivadas_oscilador.py`. Construya el arreglo de tiempos en $[0, 3]$ s con paso $\Delta t$ y el arreglo de posiciones $x(t)$.
2. Obtenga la velocidad con las cuatro fórmulas de la sección 4.1 (adelantada, atrasada, central y cinco puntos) y la aceleración con la fórmula central de la sección 4.2. Cada fórmula se evalúa solo en los puntos donde está definida: por ejemplo, la central no existe en el primer ni en el último punto.
3. Con $\Delta t = 0.01$ s, escriba `derivadas_oscilador.dat` con cuatro columnas ($t$, $x$, $v$ central, $a$ central) en los puntos donde la fórmula central está definida, y grafique $v$ y $a$ numéricas junto con las exactas en `derivadas_oscilador.png`.
4. Para $\Delta t = 0.01$ s y $\Delta t = 0.005$ s, obtenga el error máximo de cada fórmula frente a la derivada exacta. Escriba `errores_derivadas.dat` con dos filas y seis columnas: $\Delta t$, error de la adelantada, de la atrasada, de la central y de cinco puntos para $v$, y error de la central para $a$.
5. Obtenga la razón entre los errores de los dos pasos para cada fórmula y compárela con la tabla de la sección 4.2.

**Qué preparar**

| Archivo | Contenido |
|---|---|
| `derivadas_oscilador.py` | Construye los datos, deriva con rebanadas, escribe los archivos y la figura |
| `derivadas_oscilador.dat` | $\Delta t = 0.01$ s: $t$, $x$, $v$, $a$ en los 299 puntos interiores |
| `errores_derivadas.dat` | Dos filas ($\Delta t = 0.01, 0.005$): $\Delta t$ y los cinco errores máximos |
| `derivadas_oscilador.png` | $v$ y $a$ numéricas y exactas contra $t$ |

In [13]:
%%writefile derivadas_oscilador.py
# ============================================================
# Programa:  derivadas_oscilador.py
# Propósito: velocidad y aceleración de un oscilador amortiguado
#            a partir de las posiciones, con diferencias finitas
#            (rebanadas, sin ciclos sobre los puntos).
# Salida:    derivadas_oscilador.dat, errores_derivadas.dat,
#            derivadas_oscilador.png
# ============================================================
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

AZUL, DORADO, ROJO, VERDE, GRIS = "#162A54", "#C9972C", "#B02A20", "#2E7D32", "#7F7F7F"

G, W = 0.5, 2*np.pi          # gamma [1/s], omega [rad/s]
T = 3.0                      # tiempo final [s]

def x_ex(t): return np.exp(-G*t)*np.cos(W*t)
def v_ex(t): return -np.exp(-G*t)*(G*np.cos(W*t) + W*np.sin(W*t))
def a_ex(t): return np.exp(-G*t)*((G*G - W*W)*np.cos(W*t) + 2*G*W*np.sin(W*t))

def derivar(dt):
    """Devuelve la malla, las posiciones y cada derivada con sus tiempos.
    Entrada: paso dt.  Cada fórmula se evalúa solo donde está definida."""
    n = round(T/dt) + 1
    t = np.linspace(0.0, T, n)
    x = x_ex(t)
    v_adel = (x[1:] - x[:-1])/dt                                      # t[:-1]
    v_atra = (x[1:] - x[:-1])/dt                                      # t[1:]
    v_cent = (x[2:] - x[:-2])/(2*dt)                                  # t[1:-1]
    v_cinco = (-x[4:] + 8*x[3:-1] - 8*x[1:-3] + x[:-4])/(12*dt)       # t[2:-2]
    a_cent = (x[2:] - 2*x[1:-1] + x[:-2])/dt**2                       # t[1:-1]
    return t, x, v_adel, v_atra, v_cent, v_cinco, a_cent

def errores(dt):
    """Error máximo de cada fórmula frente a la derivada exacta."""
    t, x, v_adel, v_atra, v_cent, v_cinco, a_cent = derivar(dt)
    return [dt,
            np.max(np.abs(v_adel - v_ex(t[:-1]))),
            np.max(np.abs(v_atra - v_ex(t[1:]))),
            np.max(np.abs(v_cent - v_ex(t[1:-1]))),
            np.max(np.abs(v_cinco - v_ex(t[2:-2]))),
            np.max(np.abs(a_cent - a_ex(t[1:-1])))]

# --- dt = 0.01: datos en los puntos interiores y figura
dt = 0.01
t, x, v_adel, v_atra, v_cent, v_cinco, a_cent = derivar(dt)
ti = t[1:-1]
np.savetxt("derivadas_oscilador.dat",
           np.column_stack([ti, x[1:-1], v_cent, a_cent]),
           header="t x v_central a_central", fmt="%.16e")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
tf = np.linspace(0, T, 1000)
ax1.plot(tf, v_ex(tf), color=GRIS, lw=2.5, label="exacta")
ax1.plot(ti, v_cent, "--", color=AZUL, lw=1.2, label="central numérica")
ax1.set_ylabel("$v$ [m/s]"); ax1.legend()
ax2.plot(tf, a_ex(tf), color=GRIS, lw=2.5, label="exacta")
ax2.plot(ti, a_cent, "--", color=ROJO, lw=1.2, label="central numérica")
ax2.set_ylabel("$a$ [m/s$^2$]"); ax2.set_xlabel("$t$ [s]"); ax2.legend()
ax1.set_title(r"Oscilador amortiguado: derivadas numéricas ($\Delta t = 0.01$ s)")
for ax in (ax1, ax2): ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("derivadas_oscilador.png", dpi=150)

# --- errores máximos para dt = 0.01 y 0.005
E = np.array([errores(0.01), errores(0.005)])
np.savetxt("errores_derivadas.dat", E,
           header="dt err_adelantada err_atrasada err_central err_cinco_puntos err_a_central",
           fmt="%.16e")

nombres = ["adelantada", "atrasada", "central", "cinco puntos", "a central"]
razones = E[0, 1:]/E[1, 1:]
print("Errores máximos (dt = 0.01 | dt = 0.005 | razón)")
for nom, e1, e2, r in zip(nombres, E[0, 1:], E[1, 1:], razones):
    print(f"{nom:>13}: {e1:.3e} | {e2:.3e} | {r:5.2f}")

Overwriting derivadas_oscilador.py


In [14]:
# Ejecuta la Parte C (ejecute esta celda cuando su archivo esté listo)
!python derivadas_oscilador.py

Errores máximos (dt = 0.01 | dt = 0.005 | razón)
   adelantada: 1.951e-01 | 9.782e-02 |  1.99
     atrasada: 1.940e-01 | 9.756e-02 |  1.99
      central: 3.765e-03 | 9.413e-04 |  4.00
 cinco puntos: 3.028e-06 | 1.893e-07 | 15.99
    a central: 1.215e-02 | 3.082e-03 |  3.94


### D. Parte Fortran: el valor de $\pi$ con el mismo módulo

**Actividades**

1. Escriba `programa_pi.f90`, que use `modulo_romberg.f90` **sin modificarlo** para calcular $\pi = \int_0^1 \frac{4}{1+x^2}\,dx$ con tolerancia $10^{-5}$.
2. Escriba en `resultado_pi.dat` una línea con $k_{\text{final}}$, el número de evaluaciones, el valor obtenido y su error $|S_{k,k} - \pi|$.
3. Con el programa de Monte Carlo de su Taller 5, estime $\pi$ con $N = 10^6$ muestras y anote el error.
4. Con el error estadístico $\sigma/\sqrt{N}$ de Monte Carlo, donde $\sigma = 4\sqrt{p(1-p)}$ y $p = \pi/4$, obtenga cuántas muestras harían falta para un error de $10^{-5}$.
5. Resuma en el reporte una tabla con tres filas (Romberg; Monte Carlo con $10^6$ muestras; Monte Carlo con error $10^{-5}$) y dos columnas (evaluaciones, error).

**Qué preparar**

| Archivo | Contenido |
|---|---|
| `programa_pi.f90` | Usa `modulo_romberg.f90` sin cambios; escribe `resultado_pi.dat` |
| `resultado_pi.dat` | Una línea: $k_{\text{final}}$, evaluaciones, valor de $\pi$, error |

In [10]:
%%writefile programa_pi.f90
! ============================================================
! Programa:  programa_pi
! Propósito: calcular pi = integral de 4/(1+x^2) en [0, 1] con
!            la integración de Romberg (modulo_romberg sin cambios).
! Salida:    resultado_pi.dat
! ============================================================
program programa_pi
  use modulo_romberg, only: dp, pi, ID_PI, romberg
  implicit none
  real(dp), parameter :: tol = 1.0e-5_dp
  integer, parameter :: kmax = 20
  real(dp), allocatable :: S(:,:)
  real(dp) :: valor, error
  integer :: k_final, n_eval, u

  call romberg(ID_PI, 0.0_dp, 1.0_dp, tol, kmax, S, k_final, n_eval)
  valor = S(k_final, k_final)
  error = abs(valor - pi)

  open(newunit=u, file='resultado_pi.dat', status='replace')
  write(u, '(i4,1x,i6,2(1x,es24.15e3))') k_final, n_eval, valor, error
  close(u)

  print '(a,i3,a,i5)', 'k_final = ', k_final, '   evaluaciones = ', n_eval
  print '(a,f16.12)', 'pi (Romberg) = ', valor
  print '(a,es10.2)', 'error        = ', error
end program programa_pi

Writing programa_pi.f90


In [11]:
# Compila y ejecuta la Parte D (ejecute esta celda cuando sus archivos estén listos)
!gfortran -fcheck=bounds -o programa_pi modulo_romberg.f90 programa_pi.f90 && ./programa_pi

k_final =   4   evaluaciones =    17
pi (Romberg) =   3.141592665278
error        =   1.17E-08


### E. Preguntas de interpretación y reporte

Responda en el reporte, con los números de sus archivos:

1. **Conservación de la energía.** ¿Qué tan cerca están $E_{\text{dis}}$ y $E(0) - E(T)$? Compare esa diferencia con la tolerancia $10^{-6}$ y explique por qué el criterio de la diagonal es conservador.

E_dis = 18.9940046974 y E(0) − E(T) = 18.9940046969 J/kg: difieren en 4.5×10⁻¹⁰, muy por debajo de la tolerancia de 10⁻⁶. El criterio de la diagonal es conservador porque compara S₈,₈ con S₇,₇ (7.5×10⁻⁷), que mide el error del nivel anterior y no el del actual.

2. **Integrando oscilante.** La energía necesitó muchas más evaluaciones que $\pi$. Observe en la tabla de Romberg los primeros niveles (1, 2, 4 y 8 subintervalos en 3 s) y explique por qué el método no puede converger mientras la malla no resuelve las oscilaciones de $v(t)^2$.

v² tiene periodo de 0.5 s. Con 1, 2, 4 y 8 subintervalos (h = 3, 1.5, 0.75, 0.375 s) los trapecios dan 0.39, 0.28, 17.25 y 17.32, frente a 18.99. La malla no resuelve las oscilaciones (menos de dos puntos por periodo), y Romberg, que supone un error en potencias de h², extrapola mal: S₂,₂ = 24.4. Solo converge desde k = 5.

3. **Órdenes de las derivadas.** Compare las razones de la Parte C con la tabla de la sección 4.2. ¿Por qué la adelantada y la atrasada tienen errores casi iguales?

Las razones fueron 1.99, 1.99, 4.00, 15.99 y 3.94, que coinciden con 2, 2, 4, 16 y 4 de la tabla 4.2. La adelantada y la atrasada tienen errores casi iguales (0.195 y 0.194) porque comparten el término (h/2)f″ con signo contrario.

4. **Costo.** Con la tabla de la Parte D, ¿cuántas veces más evaluaciones necesita Monte Carlo que Romberg para un error de $10^{-5}$? ¿En qué situaciones sigue siendo preferible Monte Carlo?

Romberg necesita 17 evaluaciones y Monte Carlo ≈ 2.7×10¹⁰, unas 1.6×10⁹ veces más para un error de 10⁻⁵. Monte Carlo sigue siendo preferible en dimensiones altas o con regiones complicadas, porque su error no depende de la dimensión.

**Reporte (una página, plantilla del curso).** Las secciones de contexto físico, método y resultado se refieren al oscilador (Partes B y C). La Parte D puede alimentar la sección de obstáculo o decisión de implementación.

### F. Verificación

La celda siguiente **no debe modificarse**. Lee los archivos que escribieron sus programas, recalcula de forma independiente los valores de referencia, y comprueba que cada archivo es posterior al código que lo genera (un archivo viejo de una versión anterior del programa no cuenta). El taller exige los cuatro PASS.

In [15]:
# ===================== VERIFICACIÓN DEL TALLER 7 (no modificar) =====================
import os
import numpy as np

def actualizado(dat, fuentes):
    """El archivo existe y es posterior a todos los códigos fuente."""
    if not os.path.exists(dat):
        return False, f"no existe {dat}"
    for f in fuentes:
        if not os.path.exists(f):
            return False, f"no existe {f}"
        if os.path.getmtime(dat) < os.path.getmtime(f):
            return False, f"{dat} es anterior a {f}: vuelva a ejecutar el programa"
    return True, ""

G, W = 0.5, 2*np.pi
def x_ex(t): return np.exp(-G*t)*np.cos(W*t)
def v_ex(t): return -np.exp(-G*t)*(G*np.cos(W*t) + W*np.sin(W*t))
def a_ex(t): return np.exp(-G*t)*((G*G - W*W)*np.cos(W*t) + 2*G*W*np.sin(W*t))
def energia(t): return 0.5*v_ex(t)**2 + 0.5*(W*W + G*G)*x_ex(t)**2

resultados = []
def caso(nombre, funcion):
    try:
        ok, detalle = funcion()
    except Exception as e:
        ok, detalle = False, f"error al leer: {e}"
    resultados.append(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] {nombre}" + (f"  -> {detalle}" if detalle else ""))

def c1():
    ok, d = actualizado("resultado_energia.dat", ["modulo_romberg.f90", "programa_energia.f90"])
    if not ok: return ok, d
    k, n, e_dis, e_ref_est, dif = np.loadtxt("resultado_energia.dat")
    e_ref = energia(0.0) - energia(3.0)
    if abs(e_ref_est - e_ref) > 1e-10: return False, f"E(0) - E(T) = {e_ref_est:.10f}, se espera {e_ref:.10f}"
    if abs(e_dis - e_ref) >= 1e-6: return False, f"|E_dis - (E(0) - E(T))| = {abs(e_dis - e_ref):.2e}"
    if int(k) != 8 or int(n) != 257: return False, f"k_final = {int(k)}, evaluaciones = {int(n)} (se esperan 8 y 257)"
    return True, f"E_dis = {e_dis:.10f} J/kg, k = 8, 257 evaluaciones"

def c2():
    ok, d = actualizado("derivadas_oscilador.dat", ["derivadas_oscilador.py"])
    if not ok: return ok, d
    t, x, v, a = np.loadtxt("derivadas_oscilador.dat", unpack=True)
    if t.size != 299 or abs(t[1] - t[0] - 0.01) > 1e-12: return False, "se esperan 299 puntos interiores con dt = 0.01"
    if np.max(np.abs(x - x_ex(t))) > 1e-12: return False, "las posiciones no corresponden al oscilador"
    ev, ea = np.abs(v - v_ex(t)).max(), np.abs(a - a_ex(t)).max()
    if not (0.9*3.765e-3 <= ev <= 1.1*3.765e-3): return False, f"error de v = {ev:.3e}"
    if not (0.9*1.215e-2 <= ea <= 1.1*1.215e-2): return False, f"error de a = {ea:.3e}"
    return True, f"errores máximos v {ev:.3e}, a {ea:.3e}"

def c3():
    ok, d = actualizado("errores_derivadas.dat", ["derivadas_oscilador.py"])
    if not ok: return ok, d
    e = np.loadtxt("errores_derivadas.dat")
    if e.shape != (2, 6): return False, f"forma {e.shape}, se espera (2, 6)"
    if abs(e[0, 4] - 3.0275e-6)/3.0275e-6 > 0.1: return False, f"error de cinco puntos con dt = 0.01: {e[0,4]:.3e}"
    r = e[0, 1:]/e[1, 1:]
    rangos = [(1.9, 2.1), (1.9, 2.1), (3.8, 4.2), (15.0, 17.0), (3.7, 4.2)]
    if not all(lo <= ri <= hi for ri, (lo, hi) in zip(r, rangos)):
        return False, "razones " + ", ".join(f"{ri:.2f}" for ri in r)
    return True, "razones " + ", ".join(f"{ri:.2f}" for ri in r)

def c4():
    ok, d = actualizado("resultado_pi.dat", ["modulo_romberg.f90", "programa_pi.f90"])
    if not ok: return ok, d
    k, n, r, e = np.loadtxt("resultado_pi.dat")
    if abs(r - np.pi) >= 1e-5: return False, f"|R - pi| = {abs(r - np.pi):.2e}"
    if int(k) != 4 or int(n) != 17: return False, f"k_final = {int(k)}, evaluaciones = {int(n)} (se esperan 4 y 17)"
    return True, f"pi = {r:.10f}, k = 4, 17 evaluaciones"

caso("1. Energía disipada: Romberg con tol 1e-6 coincide con E(0) - E(T)", c1)
caso("2. Velocidad y aceleración centrales con dt = 0.01", c2)
caso("3. Órdenes de las cinco fórmulas de derivación", c3)
caso("4. pi con el mismo módulo: tol 1e-5, 17 evaluaciones", c4)
print(f"\nResultado: {sum(resultados)} de {len(resultados)} PASS")

[PASS] 1. Energía disipada: Romberg con tol 1e-6 coincide con E(0) - E(T)  -> E_dis = 18.9940046974 J/kg, k = 8, 257 evaluaciones
[PASS] 2. Velocidad y aceleración centrales con dt = 0.01  -> errores máximos v 3.765e-03, a 1.215e-02
[PASS] 3. Órdenes de las cinco fórmulas de derivación  -> razones 1.99, 1.99, 4.00, 15.99, 3.94
[PASS] 4. pi con el mismo módulo: tol 1e-5, 17 evaluaciones  -> pi = 3.1415926653, k = 4, 17 evaluaciones

Resultado: 4 de 4 PASS
